In [3]:
import pandas as pd
import numpy as np

# ==============================
# 1. Load Excel File
# ==============================

df = pd.read_excel("Gold_Project_Final_Corrected.xlsx")

# Clean column names
df.columns = df.columns.str.strip()

# Convert Month to Date
df["Date"] = pd.to_datetime(df["Month"], errors="coerce")

# Sort data by date
df = df.sort_values("Date").reset_index(drop=True)

# Set Date as index
df = df.set_index("Date")


# ==============================
# 2. Rename Columns for Easy Use
# ==============================

df = df.rename(columns={
    "Gold_USD (per oz)": "Gold_USD",
    "Gold Price India (10gram)": "Gold_India_10g",
    "Sensex Closing Price": "Sensex"
})


# ==============================
# 3. Create Return Variables
# ==============================

# 1. USD gold return
df["USD_Gold_Return"] = df["Gold_USD"].pct_change()

# 2. INR gold return
df["INR_Gold_Return"] = df["Gold_India_10g"].pct_change()

# 3. INR depreciation
df["INR_Depreciation"] = df["USDINR"].pct_change()

# US inflation
df["US_Inflation"] = df["US_CPI"].pct_change()

# India inflation
df["India_Inflation"] = df["India_CPI"].pct_change()

# 4. Inflation adjusted USD gold return
df["Inflation_Adjusted_USD_Gold_Return"] = (
    (1 + df["USD_Gold_Return"]) / (1 + df["US_Inflation"])
) - 1

# 5. Inflation adjusted INR gold return
df["Inflation_Adjusted_INR_Gold_Return"] = (
    (1 + df["INR_Gold_Return"]) / (1 + df["India_Inflation"])
) - 1

# 6. Inflation + depreciation adjusted INR gold return
df["Inflation_Depreciation_Adjusted_INR_Gold_Return"] = (
    (1 + df["INR_Gold_Return"]) /
    ((1 + df["India_Inflation"]) * (1 + df["INR_Depreciation"]))
) - 1

# 7. Sensex return
df["Sensex_Return"] = df["Sensex"].pct_change()

# 8. Inflation adjusted Sensex return
df["Inflation_Adjusted_Sensex_Return"] = (
    (1 + df["Sensex_Return"]) / (1 + df["India_Inflation"])
) - 1


# ==============================
# 4. CAGR Function
# ==============================

def calculate_cagr(return_series):
    """
    Calculates annual CAGR from monthly return series.
    CAGR = Product(1 + monthly returns)^(12 / number of months) - 1
    """
    s = return_series.dropna()

    n_months = len(s)

    if n_months == 0:
        return np.nan

    cumulative_return = (1 + s).prod()

    cagr = cumulative_return ** (12 / n_months) - 1

    return cagr


# ==============================
# 5. Calculate All 8 CAGR Values
# ==============================

cagr_results = {
    "USD Gold Return": calculate_cagr(df["USD_Gold_Return"]),
    "INR Gold Return": calculate_cagr(df["INR_Gold_Return"]),
    "INR Depreciation": calculate_cagr(df["INR_Depreciation"]),
    "Inflation Adjusted USD Gold Return": calculate_cagr(df["Inflation_Adjusted_USD_Gold_Return"]),
    "Inflation Adjusted INR Gold Return": calculate_cagr(df["Inflation_Adjusted_INR_Gold_Return"]),
    "Inflation + Depreciation Adjusted INR Gold Return": calculate_cagr(df["Inflation_Depreciation_Adjusted_INR_Gold_Return"]),
    "Sensex Return": calculate_cagr(df["Sensex_Return"]),
    "Inflation Adjusted Sensex Return": calculate_cagr(df["Inflation_Adjusted_Sensex_Return"])
}


# ==============================
# 6. Show CAGR Table
# ==============================

cagr_table = pd.DataFrame.from_dict(
    cagr_results,
    orient="index",
    columns=["CAGR"]
)

# Convert to percentage
cagr_table["CAGR (%)"] = (cagr_table["CAGR"] * 100).round(2)

display(cagr_table[["CAGR (%)"]])

C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_35244\1842601022.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Month"], errors="coerce")


,CAGR (%)
USD Gold Return,7.31
INR Gold Return,11.17
INR Depreciation,4.80
Inflation Adjusted USD Gold Return,4.56
Inflation Adjusted INR Gold Return,4.01
Inflation + Depreciation Adjusted INR Gold Return,-0.75
Sensex Return,13.54
Inflation Adjusted Sensex Return,6.23


In [4]:
# ── Decade-wise CAGR for all 8 series ────────────────────────────────────────

# All 8 return series
return_cols = {
    "USD Gold Return": df["USD_Gold_Return"],
    "INR Gold Return": df["INR_Gold_Return"],
    "INR Depreciation": df["INR_Depreciation"],
    "Inflation Adjusted USD Gold Return": df["Inflation_Adjusted_USD_Gold_Return"],
    "Inflation Adjusted INR Gold Return": df["Inflation_Adjusted_INR_Gold_Return"],
    "Inflation + Depreciation Adjusted INR Gold Return": df["Inflation_Depreciation_Adjusted_INR_Gold_Return"],
    "Sensex Return": df["Sensex_Return"],
    "Inflation Adjusted Sensex Return": df["Inflation_Adjusted_Sensex_Return"]
}


# CAGR function with start and end date
def calculate_cagr_period(return_series, start=None, end=None):
    """
    Calculates annual CAGR from monthly return series for a selected period.
    Formula:
    CAGR = Product(1 + monthly returns)^(12 / number of months) - 1
    """

    s = return_series.dropna()

    if start is not None:
        s = s[s.index >= start]

    if end is not None:
        s = s[s.index <= end]

    n_months = len(s)

    if n_months == 0:
        return np.nan

    cumulative_return = (1 + s).prod()
    cagr = cumulative_return ** (12 / n_months) - 1

    return cagr


# Decade periods
# Your data starts from Apr-1990, so 1990s starts from Apr-1990
decade_periods = {
    "1990s Apr 1990–Dec 1999": ("1990-04-01", "1999-12-31"),
    "2000s Jan 2000–Dec 2009": ("2000-01-01", "2009-12-31"),
    "2010s Jan 2010–Dec 2019": ("2010-01-01", "2019-12-31"),
    "2020s Jan 2020–Apr 2026": ("2020-01-01", "2026-04-30")
}


# Create decade-wise CAGR table
rows = []

for period_name, (start_date, end_date) in decade_periods.items():
    row = {"Period": period_name}

    for series_name, series_data in return_cols.items():
        row[series_name] = calculate_cagr_period(
            series_data,
            start=start_date,
            end=end_date
        )

    rows.append(row)


decade_cagr_table = pd.DataFrame(rows).set_index("Period")


# Percentage table
decade_cagr_percent = decade_cagr_table.applymap(
    lambda x: f"{x * 100:.2f}%" if pd.notnull(x) else "N/A"
)

print("=== DECADE-WISE CAGR FOR ALL 8 SERIES ===")
display(decade_cagr_percent)

=== DECADE-WISE CAGR FOR ALL 8 SERIES ===


C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_35244\2148891491.py:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  decade_cagr_percent = decade_cagr_table.applymap(


,USD Gold Return,INR Gold Return,INR Depreciation,Inflation Adjusted USD Gold Return,Inflation Adjusted INR Gold Return,Inflation + Depreciation Adjusted INR Gold Return,Sensex Return,Inflation Adjusted Sensex Return
Period,,,,,,,,
1990s Apr 1990–Dec 1999,-2.85%,2.82%,10.02%,-5.52%,-6.06%,-14.61%,20.96%,10.52%
2000s Jan 2000–Dec 2009,14.89%,14.39%,0.67%,12.03%,7.81%,7.09%,13.31%,6.79%
2010s Jan 2010–Dec 2019,2.69%,8.30%,4.34%,0.92%,1.29%,-2.92%,8.98%,1.92%
2020s Jan 2020–Apr 2026,20.21%,24.75%,4.40%,15.74%,19.70%,14.65%,10.34%,5.87%


In [5]:
# ── Pre-2021 vs Post-2021 CAGR for all 8 series ──────────────────────────────

# Periods
pre_post_periods = {
    "Pre-2021 Apr 1990–Dec 2020": ("1990-04-01", "2020-12-31"),
    "Post-2021 Jan 2021–Apr 2026": ("2021-01-01", "2026-04-30")
}

# Create Pre/Post CAGR table
rows = []

for period_name, (start_date, end_date) in pre_post_periods.items():
    row = {"Period": period_name}

    for series_name, series_data in return_cols.items():
        row[series_name] = calculate_cagr_period(
            series_data,
            start=start_date,
            end=end_date
        )

    rows.append(row)

pre_post_cagr_table = pd.DataFrame(rows).set_index("Period")

# Percentage table for display
pre_post_cagr_percent = pre_post_cagr_table.applymap(
    lambda x: f"{x * 100:.2f}%" if pd.notnull(x) else "N/A"
)

print("=== PRE-2021 VS POST-2021 CAGR FOR ALL 8 SERIES ===")
display(pre_post_cagr_percent)

=== PRE-2021 VS POST-2021 CAGR FOR ALL 8 SERIES ===


C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_35244\1314490451.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pre_post_cagr_percent = pre_post_cagr_table.applymap(


,USD Gold Return,INR Gold Return,INR Depreciation,Inflation Adjusted USD Gold Return,Inflation Adjusted INR Gold Return,Inflation + Depreciation Adjusted INR Gold Return,Sensex Return,Inflation Adjusted Sensex Return
Period,,,,,,,,
Pre-2021 Apr 1990–Dec 2020,5.36%,9.10%,4.84%,2.96%,1.65%,-3.04%,14.29%,6.47%
Post-2021 Jan 2021–Apr 2026,19.22%,23.83%,4.58%,14.26%,18.69%,13.50%,9.35%,4.82%


In [6]:
# ── Crisis-period CAGR for all 8 series ──────────────────────────────────────

# Crisis periods
crisis_periods = {
    "1991 India BoP Crisis Jan 1991–Dec 1991": ("1991-01-01", "1991-12-31"),
    "2008 Global Financial Crisis Sep 2008–Mar 2009": ("2008-09-01", "2009-03-31"),
    "COVID-19 Period Jan 2020–Dec 2021": ("2020-01-01", "2021-12-31"),
    "Post-COVID Inflation Period Jan 2022–Dec 2023": ("2022-01-01", "2023-12-31")
}

# Create Crisis-period CAGR table
rows = []

for period_name, (start_date, end_date) in crisis_periods.items():
    row = {"Period": period_name}

    for series_name, series_data in return_cols.items():
        row[series_name] = calculate_cagr_period(
            series_data,
            start=start_date,
            end=end_date
        )

    rows.append(row)

crisis_cagr_table = pd.DataFrame(rows).set_index("Period")

# Percentage table for display
crisis_cagr_percent = crisis_cagr_table.applymap(
    lambda x: f"{x * 100:.2f}%" if pd.notnull(x) else "N/A"
)

print("=== CRISIS-PERIOD CAGR FOR ALL 8 SERIES ===")
display(crisis_cagr_percent)

=== CRISIS-PERIOD CAGR FOR ALL 8 SERIES ===


C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_35244\984126522.py:29: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  crisis_cagr_percent = crisis_cagr_table.applymap(


,USD Gold Return,INR Gold Return,INR Depreciation,Inflation Adjusted USD Gold Return,Inflation Adjusted INR Gold Return,Inflation + Depreciation Adjusted INR Gold Return,Sensex Return,Inflation Adjusted Sensex Return
Period,,,,,,,,
1991 India BoP Crisis Jan 1991–Dec 1991,-4.04%,41.41%,42.43%,-6.82%,25.07%,-12.18%,82.09%,61.05%
2008 Global Financial Crisis Sep 2008–Mar 2009,18.04%,53.54%,35.07%,24.00%,48.24%,9.76%,-50.11%,-51.83%
COVID-19 Period Jan 2020–Dec 2021,10.02%,12.13%,2.93%,5.58%,7.18%,4.12%,18.83%,13.59%
Post-COVID Inflation Period Jan 2022–Dec 2023,6.38%,13.94%,5.08%,1.46%,8.30%,3.06%,11.36%,5.85%


In [7]:
# ── Year-wise CAGR after 2021 for all 8 series ───────────────────────────────

# Year-wise periods after 2021
year_periods = {
    "2021": ("2021-01-01", "2021-12-31"),
    "2022": ("2022-01-01", "2022-12-31"),
    "2023": ("2023-01-01", "2023-12-31"),
    "2024": ("2024-01-01", "2024-12-31"),
    "2025": ("2025-01-01", "2025-12-31"),
    "2026 Jan–Apr": ("2026-01-01", "2026-04-30")
}

# Create Year-wise CAGR table
rows = []

for period_name, (start_date, end_date) in year_periods.items():
    row = {"Period": period_name}

    for series_name, series_data in return_cols.items():
        row[series_name] = calculate_cagr_period(
            series_data,
            start=start_date,
            end=end_date
        )

    rows.append(row)

yearwise_cagr_table = pd.DataFrame(rows).set_index("Period")

# Percentage table for display
yearwise_cagr_percent = yearwise_cagr_table.applymap(
    lambda x: f"{x * 100:.2f}%" if pd.notnull(x) else "N/A"
)

print("=== YEAR-WISE CAGR AFTER 2021 FOR ALL 8 SERIES ===")
display(yearwise_cagr_percent)

=== YEAR-WISE CAGR AFTER 2021 FOR ALL 8 SERIES ===


C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_35244\254910118.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  yearwise_cagr_percent = yearwise_cagr_table.applymap(


,USD Gold Return,INR Gold Return,INR Depreciation,Inflation Adjusted USD Gold Return,Inflation Adjusted INR Gold Return,Inflation + Depreciation Adjusted INR Gold Return,Sensex Return,Inflation Adjusted Sensex Return
Period,,,,,,,,
2021,-3.66%,-3.14%,2.41%,-10.11%,-8.24%,-10.40%,21.99%,15.57%
2022,0.40%,12.64%,9.39%,-5.65%,6.76%,-2.40%,4.44%,-1.01%
2023,12.72%,15.26%,0.95%,9.10%,9.86%,8.83%,18.74%,13.18%
2024,30.69%,22.55%,2.06%,27.04%,18.37%,15.99%,8.17%,4.48%
2025,62.73%,72.15%,5.96%,58.61%,68.23%,58.77%,9.06%,6.58%
2026 Jan–Apr,33.54%,63.59%,11.93%,30.11%,57.60%,40.81%,-26.49%,-29.18%
